Whole-model enrichment against full HaeconGEM: hypergeom.sf(k−1,N,K,n), expected nK/N,
fold k/expected and global BH across eight primary models and all 157 categories
(1,256 tests). Enrichment requires
fold > 1 and q < 0.05.

Set the relative input/output constants in the first code cell. No local Python files
are imported.


In [ ]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
from scipy.stats import hypergeom, false_discovery_control
import cobra
from IPython.display import display

INPUT_DIR = Path("inputs")
RESULT_DIR = Path("outputs")
OUTPUT_DIR = RESULT_DIR / "whole_model_enrichment_results"
REFERENCE_PATH = INPUT_DIR / "haeconGEM1.xml"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = INPUT_DIR / "CHRR_sampling/models"
MODEL_NAMES = {
    "egg": "egg_c",
    "L1": "L1_c",
    "L2": "L2_c",
    "L3": "L3_c",
    "L4f": "L4f_c",
    "L4m": "L4m_c",
    "Af": "Af_c",
    "Am": "Am_c",
}
ALPHA = 0.05
models = {}
model_ids = {}
annotations = defaultdict(set)
names = {}
inventory = []


def save(data, name):
    d = data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
    d.to_csv(OUTPUT_DIR / name, sep="\t", index=False)
    return d


def normalize(sub):
    sub = sub or "Unassigned"
    return (
        "Aminoacyl-tRNA biosynthesis" if "Aminoacyl-tRNA biosynthesis" in sub else sub
    )


for stage, filename in MODEL_NAMES.items():
    p = MODEL_DIR / f"{filename}.xml"
    m = cobra.io.read_sbml_model(str(p))
    models[stage] = m
    model_ids[stage] = set(m.reactions.list_attr("id"))
    for r in m.reactions:
        sub = normalize(r.subsystem)
        annotations[r.id].add(sub)
        names.setdefault(r.id, r.name)
        inventory.append(
            dict(model=stage, reaction_id=r.id, name=r.name, subsystem=sub)
        )
conflicts = {
    rid: sorted(values) for rid, values in annotations.items() if len(values) != 1
}
assert not conflicts, conflicts
annotation = {rid: next(iter(values)) for rid, values in annotations.items()}
GROUP_RULES = {
    "Transport/exchange (annotation-defined)": lambda s: "transport" in s.lower()
    or s == "Exchange/demand reactions",
    "Carnitine shuttles (all compartments)": lambda s: "carnitine shuttle" in s.lower(),
    "Beta-oxidation (all annotated types)": lambda s: "beta oxidation"
    in s.lower().replace("β", "beta"),
}
save(inventory, "model_reaction_annotations.tsv")
REFERENCE_PATH = REFERENCE_PATH
full = cobra.io.read_sbml_model(str(REFERENCE_PATH))
U = set(full.reactions.list_attr("id"))
N = len(U)
full_annotation = {r.id: normalize(r.subsystem) for r in full.reactions}
missing = {stage: sorted(ids - U) for stage, ids in model_ids.items()}
save(
    [dict(model=s, reaction_id=rid) for s, ids in missing.items() for rid in ids],
    "reactions_outside_background.tsv",
)
assert not any(missing.values()), missing
save(
    [
        dict(
            reaction_id=rid,
            background_subsystem=full_annotation[rid],
            contextual_subsystem=annotation[rid],
        )
        for rid in annotation
        if annotation[rid] != full_annotation[rid]
    ],
    "annotation_differences.tsv",
)
save(
    [
        dict(reaction_id=r.id, name=r.name, subsystem=full_annotation[r.id])
        for r in full.reactions
    ],
    "background_reactions.tsv",
)


In [ ]:
categories = {
    ("subsystem", s): {rid for rid in U if full_annotation[rid] == s}
    for s in sorted(set(full_annotation.values()))
}
for group, rule in GROUP_RULES.items():
    categories["reviewer_aggregate", group] = {
        rid for rid in U if rule(full_annotation[rid])
    }
rows = []
for stage, S in model_ids.items():
    n = len(S)
    for (kind, category), ids in categories.items():
        K = len(ids)
        hits = S & ids
        k = len(hits)
        expected = n * K / N
        p = float(hypergeom.sf(k - 1, N, K, n))
        rows.append(
            dict(
                model=stage,
                category_type=kind,
                category=category,
                background_N=N,
                background_category_K=K,
                model_reactions_n=n,
                observed_k=k,
                expected=expected,
                fold_enrichment=k / expected,
                pvalue=p,
                reaction_ids=";".join(sorted(hits)),
            )
        )
results = pd.DataFrame(rows)
assert len(results) == 1256
results["q_BH_global"] = false_discovery_control(results.pvalue.to_numpy(), method="bh")
results["enriched"] = results.fold_enrichment.gt(1) & results.q_BH_global.lt(0.05)
save(results, "all_enrichment.tsv")
save(results[results.category_type == "subsystem"], "subsystem_enrichment.tsv")
agg = save(
    results[results.category_type == "reviewer_aggregate"],
    "reviewer_categories_enrichment.tsv",
)
summary = (
    results.groupby("model", sort=False)
    .agg(
        model_reactions=("model_reactions_n", "first"),
        tests=("category", "size"),
        enriched_tests=("enriched", "sum"),
    )
    .reset_index()
)
save(summary, "model_summary.tsv")
save(
    [
        dict(model=stage, reaction_id=rid, subsystem=full_annotation[rid])
        for stage, S in model_ids.items()
        for rid in sorted(S)
    ],
    "model_reaction_membership.tsv",
)
display(agg.drop(columns=["reaction_ids"]).round(6))
